# Binance Demo Spot - World Model Data Collection

Notebook này thu thập dữ liệu cho world-model theo Spot Demo Mode của Binance.

Mục tiêu:
- Chọn 30 cặp USDT thanh khoản cao nhất
- Chia thành 3 batch, mỗi batch 10 cặp
- Chạy một lần notebook để đi qua toàn bộ 3 batch
- Append vào `asset_klines.csv` và deduplicate theo `symbol + interval + open_time`

File đầu ra:
- `asset_klines.csv`
- `benchmark_klines.csv`
- `portfolio_positions.csv`
- `top_20_usdt_symbols.csv`

In [1]:
# Cell 1 - Imports

import os
import time
import hmac
import hashlib
from urllib.parse import urlencode
from typing import Dict, List, Optional, Tuple

import requests
import pandas as pd

In [2]:
# Cell 2 - Configuration

API_KEY = os.getenv("BINANCE_DEMO_API_KEY")
API_SECRET = os.getenv("BINANCE_DEMO_API_SECRET")

BASE_URL = "https://demo-api.binance.com/api"

TARGET_QUOTE_ASSET = "USDT"
TOP_N_SYMBOLS = 20
BATCH_SIZE = 10
BATCH_INDICES = [0, 1]

INTERVAL = "15m"
START_TIME = "2026-02-01 00:00:00"
END_TIME = "2026-04-07 23:59:59"

BENCHMARK_SYMBOLS = ["BTCUSDT", "ETHUSDT"]

OMIT_ZERO_BALANCES = True

OUTPUT_DIR = "./data_raw"
os.makedirs(OUTPUT_DIR, exist_ok=True)

print("Output directory:", OUTPUT_DIR)

Output directory: ./data_raw


In [13]:
# Cell 3 - Utility functions

def to_millis(dt_str: str) -> int:
    return int(pd.Timestamp(dt_str, tz="UTC").timestamp() * 1000)

def from_millis(ms: int) -> pd.Timestamp:
    return pd.to_datetime(ms, unit="ms", utc=True)

def sign_params(params: Dict[str, object], api_secret: str) -> str:
    query_string = urlencode(params, doseq=True)
    return hmac.new(
        api_secret.encode("utf-8"),
        query_string.encode("utf-8"),
        hashlib.sha256
    ).hexdigest()

def save_append_dedup(
    new_df: pd.DataFrame,
    csv_path: str,
    dedup_subset: List[str],
    sort_cols: List[str],
    parse_dates: Optional[List[str]] = None
) -> pd.DataFrame:
    if os.path.exists(csv_path):
        existing_df = pd.read_csv(csv_path, parse_dates=parse_dates)
        combined_df = pd.concat([existing_df, new_df], ignore_index=True)
    else:
        combined_df = new_df.copy()

    combined_df = (
        combined_df
        .drop_duplicates(subset=dedup_subset)
        .sort_values(sort_cols)
        .reset_index(drop=True)
    )
    combined_df.to_csv(csv_path, index=False)
    return combined_df

def make_request(method, path, params=None, headers=None, signed=False, max_retries=3, timeout=60):
    url = f"{BASE_URL}{path}"
    if params is None: params = {}
    params = dict(params)
    if signed:
        params["timestamp"] = int(time.time() * 1000)
        params["recvWindow"] = 5000
        params["signature"] = sign_params(params, API_SECRET)
        if headers is None: headers = {}
        headers["X-MBX-APIKEY"] = API_KEY
    last_err = None
    for attempt in range(max_retries):
        try:
            r = requests.request(method=method, url=url, params=params, headers=headers, timeout=timeout)
            r.raise_for_status()
            return r.json()
        except (requests.exceptions.ReadTimeout, requests.exceptions.ConnectionError) as e:
            last_err = e
            wait_time = (attempt + 1) * 2
            print(f"  [Attempt {attempt+1}/{max_retries}] Lỗi: {e}. Thử lại sau {wait_time}s...")
            time.sleep(wait_time)
        except requests.exceptions.HTTPError as e:
            if e.response.status_code == 429: # Rate Limit
                retry_after = int(e.response.headers.get("Retry-After", 60))
                print(f"  [Rate Limit] Đợi {retry_after}s...")
                time.sleep(retry_after)
                continue
            raise e
    raise last_err
    
def public_get(path, params=None):
    return make_request("GET", path, params=params, signed=False)
    
def signed_get(path, params=None):
    return make_request("GET", path, params=params, signed=True)

In [12]:
# Cell 4 - Endpoint sanity check

server_time = public_get("/v3/time")
exchange_info = public_get("/v3/exchangeInfo")

print("Server time:", from_millis(server_time["serverTime"]))
print("Number of exchange symbols:", len(exchange_info.get("symbols", [])))

Server time: 2026-04-07 18:45:45.725000+00:00
Number of exchange symbols: 3556


In [5]:
# Cell 5 - Klines downloader

def fetch_klines(symbol: str, interval: str, start_ms: int, end_ms: int, limit: int = 1000) -> pd.DataFrame:
    all_rows = []
    current_start = start_ms

    while current_start < end_ms:
        params = {
            "symbol": symbol,
            "interval": interval,
            "startTime": current_start,
            "endTime": end_ms,
            "limit": limit,
        }
        rows = public_get("/v3/klines", params=params)

        if not rows:
            break

        all_rows.extend(rows)

        last_open_time = rows[-1][0]
        next_start = last_open_time + 1

        if next_start <= current_start:
            break

        current_start = next_start
        time.sleep(0.08)

    if not all_rows:
        return pd.DataFrame()

    cols = [
        "open_time",
        "open",
        "high",
        "low",
        "close",
        "volume",
        "close_time",
        "quote_volume",
        "trade_count",
        "taker_buy_base_volume",
        "taker_buy_quote_volume",
        "ignore",
    ]
    df = pd.DataFrame(all_rows, columns=cols)

    numeric_cols = [
        "open", "high", "low", "close", "volume",
        "quote_volume", "taker_buy_base_volume", "taker_buy_quote_volume"
    ]
    for c in numeric_cols:
        df[c] = pd.to_numeric(df[c], errors="coerce")

    df["trade_count"] = pd.to_numeric(df["trade_count"], errors="coerce").astype("Int64")
    df["open_time"] = pd.to_datetime(df["open_time"], unit="ms", utc=True)
    df["close_time"] = pd.to_datetime(df["close_time"], unit="ms", utc=True)

    df = df.drop(columns=["ignore"]).drop_duplicates(subset=["open_time"]).sort_values("open_time").reset_index(drop=True)
    df["symbol"] = symbol
    df["interval"] = interval

    ordered_cols = [
        "symbol", "interval", "open_time", "close_time",
        "open", "high", "low", "close",
        "volume", "quote_volume", "trade_count",
        "taker_buy_base_volume", "taker_buy_quote_volume",
    ]
    return df[ordered_cols]

In [ ]:
# Cell 6 - Build fixed top 30 liquid USDT universe

def get_top_liquid_usdt_symbols(top_n: int = 30, quote_asset: str = "USDT") -> Tuple[List[str], pd.DataFrame]:
    exchange_info = public_get("/v3/exchangeInfo")
    ticker_24h = public_get("/v3/ticker/24hr")

    symbols_info = pd.DataFrame(exchange_info["symbols"])
    ticker_df = pd.DataFrame(ticker_24h)

    symbols_info = symbols_info.loc[
        (symbols_info["status"] == "TRADING") &
        (symbols_info["quoteAsset"] == quote_asset)
    ].copy()

    merged = symbols_info.merge(
        ticker_df[["symbol", "quoteVolume", "volume", "lastPrice"]],
        on="symbol",
        how="left"
    )

    merged["quoteVolume"] = pd.to_numeric(merged["quoteVolume"], errors="coerce")
    merged["volume"] = pd.to_numeric(merged["volume"], errors="coerce")
    merged["lastPrice"] = pd.to_numeric(merged["lastPrice"], errors="coerce")

    exclude_keywords = ["UPUSDT", "DOWNUSDT", "BULLUSDT", "BEARUSDT"]
    for kw in exclude_keywords:
        merged = merged.loc[~merged["symbol"].str.contains(kw, na=False)]

    merged = merged.sort_values("quoteVolume", ascending=False).reset_index(drop=True)
    top_symbols = merged["symbol"].head(top_n).tolist()

    return top_symbols, merged

TOP_SYMBOLS, SYMBOL_LIQUIDITY_TABLE = get_top_liquid_usdt_symbols(
    top_n=TOP_N_SYMBOLS,
    quote_asset=TARGET_QUOTE_ASSET
)

print("Top liquid USDT symbols:")
print(TOP_SYMBOLS)

top_20_path = os.path.join(OUTPUT_DIR, "top_20_usdt_symbols.csv")
pd.DataFrame({
    "rank": range(1, len(TOP_SYMBOLS) + 1),
    "symbol": TOP_SYMBOLS
}).to_csv(top_20_path, index=False)

print("Saved:", top_20_path)
display(SYMBOL_LIQUIDITY_TABLE.head())

In [7]:
# Cell 7 - Batch overview

def get_batch(symbols: List[str], batch_size: int, batch_index: int) -> List[str]:
    start = batch_index * batch_size
    end = start + batch_size
    return symbols[start:end]

batch_table = []
for batch_idx in BATCH_INDICES:
    batch_symbols = get_batch(TOP_SYMBOLS, BATCH_SIZE, batch_idx)
    batch_table.append({
        "batch_index": batch_idx,
        "num_symbols": len(batch_symbols),
        "symbols": ", ".join(batch_symbols)
    })

batch_df = pd.DataFrame(batch_table)
display(batch_df)

,batch_index,num_symbols,symbols
0,0,10,"USDCUSDT, BTCUSDT, QKCUSDT, ETHUSDT, WANUSDT, ..."
1,1,10,"TAOUSDT, ZECUSDT, BNBUSDT, GNOUSDT, DOGEUSDT, ..."


In [10]:
# Cell 8 - Run all asset batches, save after each batch, stop on failed batch

start_ms = to_millis(START_TIME)
end_ms = to_millis(END_TIME)

asset_klines_path = os.path.join(OUTPUT_DIR, "asset_klines.csv")

batch_run_log = []

for batch_idx in BATCH_INDICES:
    batch_symbols = get_batch(TOP_SYMBOLS, BATCH_SIZE, batch_idx)
    if not batch_symbols:
        print(f"Skipping empty batch {batch_idx}")
        continue

    print("=" * 80)
    print(f"Running batch {batch_idx} with {len(batch_symbols)} symbols")
    print(batch_symbols)

    try:
        batch_frames = []
        for symbol in batch_symbols:
            print(f"Fetching asset klines for {symbol} ...")
            df_symbol = fetch_klines(symbol, INTERVAL, start_ms, end_ms)
            print(f"  rows fetched: {len(df_symbol)}")
            batch_frames.append(df_symbol)

        batch_new_df = pd.concat(batch_frames, ignore_index=True) if batch_frames else pd.DataFrame()

        combined_asset_klines = save_append_dedup(
            new_df=batch_new_df,
            csv_path=asset_klines_path,
            dedup_subset=["symbol", "interval", "open_time"],
            sort_cols=["symbol", "open_time"],
            parse_dates=["open_time", "close_time"]
        )

        batch_run_log.append({
            "batch_index": batch_idx,
            "status": "success",
            "symbols_count": len(batch_symbols),
            "rows_fetched_this_batch": len(batch_new_df),
            "total_rows_after_save": len(combined_asset_klines)
        })

        print(f"Saved batch {batch_idx} -> {asset_klines_path}")
        print(f"Rows fetched this batch: {len(batch_new_df)}")
        print(f"Total rows after save: {len(combined_asset_klines)}")

    except Exception as e:
        batch_run_log.append({
            "batch_index": batch_idx,
            "status": f"failed: {str(e)}",
            "symbols_count": len(batch_symbols),
            "rows_fetched_this_batch": None,
            "total_rows_after_save": None
        })
        print(f"Batch {batch_idx} failed with error: {e}")
        break

print("=" * 80)
print("Batch run summary:")
display(pd.DataFrame(batch_run_log))

if os.path.exists(asset_klines_path):
    latest_asset_klines = pd.read_csv(asset_klines_path, parse_dates=["open_time", "close_time"])
    display(latest_asset_klines.tail())

Running batch 0 with 10 symbols
['USDCUSDT', 'BTCUSDT', 'QKCUSDT', 'ETHUSDT', 'WANUSDT', 'SOLUSDT', 'BIFIUSDT', 'USD1USDT', 'FARMUSDT', 'XRPUSDT']
Fetching asset klines for USDCUSDT ...
  rows fetched: 6314
Fetching asset klines for BTCUSDT ...
  rows fetched: 6314
Fetching asset klines for QKCUSDT ...
  rows fetched: 6314
Fetching asset klines for ETHUSDT ...
  rows fetched: 6314
Fetching asset klines for WANUSDT ...
  rows fetched: 6314
Fetching asset klines for SOLUSDT ...
  rows fetched: 6314
Fetching asset klines for BIFIUSDT ...
  rows fetched: 6314
Fetching asset klines for USD1USDT ...
  rows fetched: 6314
Fetching asset klines for FARMUSDT ...
  rows fetched: 6314
Fetching asset klines for XRPUSDT ...
  rows fetched: 6314
Saved batch 0 -> ./data_raw\asset_klines.csv
Rows fetched this batch: 63140
Total rows after save: 63140
Running batch 1 with 10 symbols
['TAOUSDT', 'GNOUSDT', 'ZECUSDT', 'BNBUSDT', 'DOGEUSDT', 'AVAXUSDT', 'REDUSDT', 'EURUSDT', 'XAUTUSDT', 'PEPEUSDT']
Fetchin

,batch_index,status,symbols_count,rows_fetched_this_batch,total_rows_after_save
0,0,success,10,63140.0,63140.0
1,1,success,10,57996.0,121136.0
2,2,failed: HTTPSConnectionPool(host='demo-api.bin...,10,NaN,NaN


,symbol,interval,open_time,close_time,open,high,low,close,volume,quote_volume,trade_count,taker_buy_base_volume,taker_buy_quote_volume
121131,ZECUSDT,15m,2026-04-07 17:15:00+00:00,2026-04-07 17:29:59.999000+00:00,269.48,270.07,267.50,268.56,1540.094,4.133425e+05,1239,641.913,172410.00763
121132,ZECUSDT,15m,2026-04-07 17:30:00+00:00,2026-04-07 17:44:59.999000+00:00,268.57,273.99,268.57,271.45,4462.082,1.213703e+06,4063,2379.051,647006.97961
121133,ZECUSDT,15m,2026-04-07 17:45:00+00:00,2026-04-07 17:59:59.999000+00:00,271.47,272.91,269.28,270.30,6385.842,1.731734e+06,2531,1947.202,528356.53646
121134,ZECUSDT,15m,2026-04-07 18:00:00+00:00,2026-04-07 18:14:59.999000+00:00,270.30,270.37,269.03,270.00,871.558,2.351140e+05,881,511.283,137966.83766
121135,ZECUSDT,15m,2026-04-07 18:15:00+00:00,2026-04-07 18:29:59.999000+00:00,269.96,270.35,269.05,269.15,578.592,1.560697e+05,509,247.161,66653.34050


In [14]:
# Cell 9 - Build / append benchmark_klines.csv

start_ms = to_millis(START_TIME)
end_ms = to_millis(END_TIME)

benchmark_klines_path = os.path.join(OUTPUT_DIR, "benchmark_klines.csv")

benchmark_frames = []
for symbol in BENCHMARK_SYMBOLS:
    print(f"Fetching benchmark klines for {symbol} ...")
    df_symbol = fetch_klines(symbol, INTERVAL, start_ms, end_ms)
    print(f"  rows fetched: {len(df_symbol)}")
    benchmark_frames.append(df_symbol)

new_benchmark_klines = pd.concat(benchmark_frames, ignore_index=True) if benchmark_frames else pd.DataFrame()

combined_benchmark_klines = save_append_dedup(
    new_df=new_benchmark_klines,
    csv_path=benchmark_klines_path,
    dedup_subset=["symbol", "interval", "open_time"],
    sort_cols=["symbol", "open_time"],
    parse_dates=["open_time", "close_time"]
)

print("Saved / updated:", benchmark_klines_path)
print("New rows fetched:", len(new_benchmark_klines))
print("Total rows after concat:", len(combined_benchmark_klines))
display(combined_benchmark_klines.tail())

Fetching benchmark klines for BTCUSDT ...
  rows fetched: 6316
Fetching benchmark klines for ETHUSDT ...
  rows fetched: 6316
Saved / updated: ./data_raw\benchmark_klines.csv
New rows fetched: 12632
Total rows after concat: 12632


,symbol,interval,open_time,close_time,open,high,low,close,volume,quote_volume,trade_count,taker_buy_base_volume,taker_buy_quote_volume
12627,ETHUSDT,15m,2026-04-07 17:45:00+00:00,2026-04-07 17:59:59.999000+00:00,2097.22,2102.55,2096.80,2099.50,2545.3128,5.344981e+06,5039,1680.9514,3.529897e+06
12628,ETHUSDT,15m,2026-04-07 18:00:00+00:00,2026-04-07 18:14:59.999000+00:00,2099.49,2101.39,2094.33,2096.93,1606.1869,3.368266e+06,4386,715.2767,1.500334e+06
12629,ETHUSDT,15m,2026-04-07 18:15:00+00:00,2026-04-07 18:29:59.999000+00:00,2096.93,2096.94,2093.48,2094.80,1377.3807,2.886095e+06,3359,769.9680,1.613331e+06
12630,ETHUSDT,15m,2026-04-07 18:30:00+00:00,2026-04-07 18:44:59.999000+00:00,2094.79,2098.14,2090.18,2090.91,1271.7638,2.662467e+06,4002,509.8033,1.067456e+06
12631,ETHUSDT,15m,2026-04-07 18:45:00+00:00,2026-04-07 18:59:59.999000+00:00,2090.91,2091.01,2089.29,2091.00,340.9392,7.125848e+05,609,85.1330,1.779359e+05


In [ ]:
# Cell 12 - Final validation summary

asset_klines = pd.read_csv(
    os.path.join(OUTPUT_DIR, "asset_klines.csv"),
    parse_dates=["open_time", "close_time"]
)
benchmark_klines = pd.read_csv(
    os.path.join(OUTPUT_DIR, "benchmark_klines.csv"),
    parse_dates=["open_time", "close_time"]
)

print("asset_klines rows:", len(asset_klines))
print("benchmark_klines rows:", len(benchmark_klines))

print("\nUnique asset symbols collected:", asset_klines["symbol"].nunique())
print(sorted(asset_klines["symbol"].unique().tolist()))

print("\nUnique benchmark symbols:", benchmark_klines["symbol"].nunique())
print(sorted(benchmark_klines["symbol"].unique().tolist()))

print("\nHead of asset_klines:")
display(asset_klines.head())

print("\nHead of benchmark_klines:")
display(benchmark_klines.head())